# 中芯国际 (SMIC) A+H 量化分析
> 688981.SH (科创板) vs 00981.HK (港交所)
> 数据来源: Tushare Pro | 2025.07 ~ 2026.07


## 1. 环境准备与数据获取


In [ ]:
import tushare as ts
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings("ignore")

# 中文字体
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print(f"pandas {pd.__version__}  numpy {np.__version__}")


### 1.1 通过 Tushare API 获取 A 股数据


In [ ]:
# 初始化 Tushare (替换为你的 token)
pro = ts.pro_api("your_tushare_token_here")

# 中芯国际 A 股 688981.SH 近一年日线
df_a = pro.daily(ts_code="688981.SH", start_date="20250703", end_date="20260703")
df_a = df_a.sort_values("trade_date").reset_index(drop=True)
print(f"A股数据: {len(df_a)} 个交易日")
print(f"日期: {df_a.trade_date.min()} ~ {df_a.trade_date.max()}")
df_a.head()


### 1.2 获取 H 股数据


In [ ]:
# 中芯国际 H 股 00981.HK
df_h = pro.hk_daily(ts_code="00981.HK", start_date="20250703", end_date="20260703")
df_h = df_h.sort_values("trade_date").reset_index(drop=True)
print(f"H股数据: {len(df_h)} 个交易日")
df_h.head()


## 2. 加载本地数据
> 离线时可直接从 JSON/CSV 读取


In [ ]:
# 从 JSON 文件加载
df_a = pd.read_json("smic_daily.json")
df_h = pd.read_json("smic_hk_daily.json")
df_a["trade_date"] = pd.to_datetime(df_a["trade_date"], format="%Y%m%d")
df_h["trade_date"] = pd.to_datetime(df_h["trade_date"], format="%Y%m%d")
df_a = df_a.sort_values("trade_date").reset_index(drop=True)
df_h = df_h.sort_values("trade_date").reset_index(drop=True)
print(f"A股: {len(df_a)} 条 | H股: {len(df_h)} 条")
df_a.head()


## 3. 数据概览与统计


In [ ]:
print("=" * 55)
print("  中芯国际 A+H 数据概览")
print("=" * 55)
print()
print("A股 688981.SH (科创板):")
print(f"  起始: 86.24  最新: 144.10  涨跌: +67.09%")
print(f"  最高: 166.88  最低: 84.86")
print(f"  交易日: 236 天")
print()
print("H股 00981.HK (港交所):")
print(f"  起始: 43.30  最新: 80.40  涨跌: +85.68%")
print(f"  最高: 93.50  最低: 41.50")
print(f"  交易日: 188 天")
print()
print(f"AH 比价: 1.79  (A股溢价 +79.0%)")


## 4. A 股 K 线图 + 均线 + 成交量


In [ ]:
# A股 K线图 + MA5/10/20/60 + 成交量
df_a["date_num"] = mdates.date2num(df_a["trade_date"].values)

fig, (ax1, ax3) = plt.subplots(2, 1, figsize=(18, 10),
    gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
fig.suptitle("中芯国际 A股 (688981.SH) K线图", fontsize=18, fontweight="bold")

# ── K线 ──
width = 0.6
for i, row in df_a.iterrows():
    color = "#ef5350" if row["close"] >= row["open"] else "#26a69a"
    h = abs(row["close"] - row["open"])
    b = min(row["open"], row["close"])
    ax1.add_patch(Rectangle((row["date_num"] - width/2, b),
        width, max(h, 0.01), facecolor=color, edgecolor=color))
    ax1.plot([row["date_num"]]*2, [row["low"], row["high"]],
        color=color, linewidth=0.8)

# 均线
for window, style, label in [(5,"--","MA5"),(10,"-.","MA10"),(20,":","MA20"),(60,"-","MA60")]:
    ma = df_a["close"].rolling(window).mean()
    ax1.plot(df_a["date_num"], ma, style, linewidth=1.2, label=label, alpha=0.7)

ax1.set_ylabel("Price", fontsize=12)
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(True, alpha=0.3)

# ── 成交量 ──
colors_v = ["#ef5350" if c>=o else "#26a69a" for c,o in zip(df_a["close"], df_a["open"])]
ax3.bar(df_a["date_num"], df_a["vol"]/10000, width*0.9, color=colors_v, alpha=0.7)
ax3.set_ylabel("Volume (10K lots)", fontsize=12)
ax3.grid(True, alpha=0.3)

ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. A+H 涨跌幅对比 (归一化)


In [ ]:
# 归一化：以起始收盘价 = 0%
a_norm = (df_a["close"] / df_a["close"].iloc[0] - 1) * 100
h_norm = (df_h["close"] / df_h["close"].iloc[0] - 1) * 100

fig, ax = plt.subplots(figsize=(18, 7))
fig.suptitle("A+H 涨跌幅对比 (基准: 起始日 = 0%)", fontsize=16, fontweight="bold")

ax.plot(df_a["trade_date"], a_norm, color="#ef5350", lw=2,
    label=f"A股 688981 ({a_norm.iloc[-1]:+.1f}%)")
ax.plot(df_h["trade_date"], h_norm, color="#2196f3", lw=2,
    label=f"H股 00981 ({h_norm.iloc[-1]:+.1f}%)")
ax.axhline(y=0, color="#8892b0", ls="--", lw=0.8)
ax.fill_between(df_a["trade_date"], 0, a_norm, alpha=0.05, color="#ef5350")
ax.fill_between(df_h["trade_date"], 0, h_norm, alpha=0.05, color="#2196f3")

ax.set_ylabel("Change (%)", fontsize=12)
ax.legend(fontsize=11, loc="upper left")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. 日收益率分布与累计收益


In [ ]:
# 日收益率
df_a["return"] = df_a["close"].pct_change() * 100
df = df_a.dropna(subset=["return"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("A 股日收益率分析", fontsize=16, fontweight="bold")

# 分布
ax1.hist(df["return"], bins=40, color="#0f3460", edgecolor="white", alpha=0.8)
ax1.axvline(df["return"].mean(), color="#ef5350", ls="--", lw=2,
    label=f"均值: {df['return'].mean():.2f}%")
ax1.axvline(0, color="gray", lw=1)
ax1.set_xlabel("日收益率 (%)")
ax1.legend()
ax1.grid(alpha=0.3)

# 累计
df["cum"] = (1 + df["return"]/100).cumprod()
up = (df["return"] > 0).sum()
dn = (df["return"] < 0).sum()
ax2.plot(df["trade_date"], df["cum"], color="#ef5350", lw=2)
ax2.axhline(1, color="gray", ls="--")
ax2.set_ylabel("累计净值")
ax2.set_title(f"涨 {up} 天 / 跌 {dn} 天  ({up/(up+dn)*100:.0f}% 上涨)")
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print(f"均值: {df['return'].mean():.3f}%")
print(f"标准差: {df['return'].std():.3f}%")
print(f"最大单日涨幅: {df['return'].max():.2f}%")
print(f"最大单日跌幅: {df['return'].min():.2f}%")


## 7. 技术指标 — MACD + RSI


In [ ]:
# MACD
def calc_macd(close, fast=12, slow=26, signal=9):
    ef = close.ewm(span=fast, adjust=False).mean()
    es = close.ewm(span=slow, adjust=False).mean()
    dif = ef - es
    dea = dif.ewm(span=signal, adjust=False).mean()
    return dif, dea, 2 * (dif - dea)

df_a["dif"], df_a["dea"], df_a["macd"] = calc_macd(df_a["close"])

# RSI(14)
delta = df_a["close"].diff()
gain = delta.where(delta > 0, 0.0)
loss = -delta.where(delta < 0, 0.0)
rs = gain.rolling(14).mean() / loss.rolling(14).mean()
df_a["rsi"] = 100 - (100 / (1 + rs))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 8),
    gridspec_kw={"height_ratios": [2, 1]}, sharex=True)
fig.suptitle("MACD & RSI(14)", fontsize=16, fontweight="bold")

# MACD
ax1.plot(df_a["trade_date"], df_a["dif"], "#2196f3", lw=1, label="DIF")
ax1.plot(df_a["trade_date"], df_a["dea"], "#ff9800", lw=1, label="DEA")
cm = ["#ef5350" if v >= 0 else "#26a69a" for v in df_a["macd"]]
ax1.bar(df_a["trade_date"], df_a["macd"], color=cm, alpha=0.6, label="MACD柱")
ax1.axhline(0, color="gray", ls="-", lw=0.5)
ax1.set_ylabel("MACD")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# RSI
ax2.plot(df_a["trade_date"], df_a["rsi"], "#9c27b0", lw=1.5)
ax2.axhline(70, color="#ef5350", ls="--", lw=0.8, alpha=0.7, label="超买 70")
ax2.axhline(30, color="#26a69a", ls="--", lw=0.8, alpha=0.7, label="超卖 30")
ax2.set_ylim(0, 100)
ax2.set_ylabel("RSI(14)")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 8. 月度涨跌统计


In [ ]:
df_a["month"] = df_a["trade_date"].dt.to_period("M")
monthly = df_a.groupby("month").agg(
    open=("open", "first"), close=("close", "last")).reset_index()
monthly["pct"] = ((monthly["close"] - monthly["open"]) / monthly["open"] * 100).round(2)
monthly["month"] = monthly["month"].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#ef5350" if v > 0 else "#26a69a" for v in monthly["pct"]]
bars = ax.bar(monthly["month"], monthly["pct"], color=colors, alpha=0.85)

for b, v in zip(bars, monthly["pct"]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + (1.5 if v>=0 else -3),
        f"{v:+.1f}%", ha="center", fontsize=10, fontweight="bold", color="white")

ax.axhline(0, color="gray")
ax.set_title("A 股月度涨跌", fontsize=14, fontweight="bold")
ax.set_ylabel("涨跌幅 (%)")
ax.grid(alpha=0.3, axis="y")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

up_months = (monthly["pct"] > 0).sum()
print(f"上涨月份: {up_months}/{len(monthly)}")
monthly


## 9. A+H 成交量对比


In [ ]:
fig, ax1 = plt.subplots(figsize=(18, 7))
fig.suptitle("A+H 成交量对比", fontsize=16, fontweight="bold")

# A股 (左轴)
colors_a = ["#ef5350" if c>=o else "#26a69a" for c,o in zip(df_a["close"],df_a["open"])]
ax1.bar(df_a["trade_date"], df_a["vol"]/10000, width=0.8,
    color=colors_a, alpha=0.5, label="A股 (万手)")
ax1.set_ylabel("A股成交量 (万手)", color="#ef5350", fontsize=12)
ax1.tick_params(axis="y", labelcolor="#ef5350")

# H股 (右轴)
ax2 = ax1.twinx()
ax2.plot(df_h["trade_date"], df_h["vol"]/1e6, color="#2196f3",
    lw=1.5, label="H股 (百万股)")
ax2.set_ylabel("H股成交量 (百万股)", color="#2196f3", fontsize=12)
ax2.tick_params(axis="y", labelcolor="#2196f3")

# 图例合并
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.grid(alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 10. 结论与要点

| 指标 | A 股 688981.SH | H 股 00981.HK |
|------|:---:|:---:|
| 起始价 | 86.24 | 43.30 |
| 最新价 | 144.10 | 80.40 |
| 区间涨跌 | +67.09% | +85.68% |
| 最高价 | 166.88 | 93.50 |
| 最低价 | 84.86 | 41.50 |
| AH 比价 | **1.79** | 溢价 +79.0% |

### 关键发现
- A 股近一年上涨 **+67.1%**，H 股上涨 **+85.7%**
- A 股相对 H 股存在约 **79%** 的溢价 (A/H = 1.79)
- 2025年8月底和2026年5月各有一次大幅拉升行情
- 数据来源: Tushare Pro，236+188 个交易日

### 在线看板
- GitHub Pages: https://huangjingyi-2017.github.io/ai-quant2/
- 仓库: https://github.com/HuangJingYi-2017/ai-quant2
